### IMPORTS DEPENDENCIES

In [3]:
import openai
import instructor
import qdrant_client
from qdrant_client  import QdrantClient

from pydantic import BaseModel, Field

from apps.chatbot_ui.src.chatbot_ui.app import output



In [4]:
prompt = """
You are a helpful assistant.
Return an answer to the question.
Question: What is your name? How do u differ from Gemini model?
"""

In [5]:
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": prompt},
    ],
    temperature=0
)

print(response.choices[0].message.content)

I am ChatGPT, an AI language model developed by OpenAI. 

Regarding how I differ from the Gemini model: Gemini is a language model developed by Google DeepMind, while I am based on OpenAI's GPT architecture. Although both are advanced AI language models designed to understand and generate human-like text, they differ in their underlying architectures, training data, and specific capabilities as determined by their respective organizations. Each model may have unique strengths depending on its design goals, training methods, and updates.


In [6]:
client = instructor.from_openai(openai.OpenAI())

In [18]:
class RAGGenerationResponse(BaseModel):
    answer:str = Field(description="The answer to the question.")

In [22]:
response, row_response = client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    temperature=0,
    response_model=RAGGenerationResponse
)



In [24]:
row_response

ChatCompletion(id='chatcmpl-DTrGgfjhmAFMYz0Egr7yX0ADpp2Em', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_KVZQuBM3Raw6PwIGdciJNVor', function=Function(arguments='{"answer":"My name is ChatGPT. I am an AI language model developed by OpenAI. The Gemini model, developed by Google DeepMind, is another advanced AI language model. While both models are designed to understand and generate human-like text, they differ in their underlying architectures, training data, and specific capabilities. ChatGPT is based on the GPT architecture, whereas Gemini incorporates Google\'s research advancements. Each model has unique strengths depending on its design and intended applications."}', name='RAGGenerationResponse'), type='function')]))], created=1776008646, model='gpt-4.1-mini-2025-04-14', object='chat.c

In [25]:
class RAGGenerationResponse(BaseModel):
    answer:str = Field(description="The answer to the question.")
    reasoning: str = Field(description="The reasoning about the answer.")

In [26]:
response, row_response = client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    temperature=0,
    response_model=RAGGenerationResponse
)

In [27]:
response

RAGGenerationResponse(answer="My name is ChatGPT. I am an AI language model developed by OpenAI. The Gemini model, developed by Google DeepMind, is another advanced AI language model. While both models are designed to understand and generate human-like text, they differ in their underlying architectures, training data, and specific capabilities. For example, Gemini may have different strengths in certain tasks or integrations with Google's ecosystem, whereas I am optimized for a wide range of conversational and informational tasks across various domains.", reasoning='The question asks for my name and how I differ from the Gemini model. I provided my name as ChatGPT and explained that Gemini is a different AI model developed by Google DeepMind. I highlighted that differences arise from their architectures, training, and capabilities, which is a common distinction between AI models from different organizations.')

In [28]:
class RAGGenerationResponse(BaseModel):
    answer:str = Field(description="The answer to the question.")


In [42]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-item-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt


## This changes, see the instructor response model is added here (Pydantic inheritance from BaseModel)
def generate_answer(prompt):

    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role": "system", "content": prompt}],
        temperature=0,
        response_model=RAGGenerationResponse
    )

    return response, raw_response


def rag_pipeline(question, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer, raw_response = generate_answer(prompt)

    final_result = {
        ## The RAGGenerationResponse data model
        "datamodel": answer,
        ## Actual answer
        "answer": answer.answer,
        "rawresponse": raw_response,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result

In [39]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [43]:
output = rag_pipeline("Can I get a charging cable? Please suggest me a good one.", qdrant_client)

In [44]:
output

{'datamodel': RAGGenerationResponse(answer='Based on the available products, I suggest the GREPHONE 2 Pack USB C to Lightning Cable, 6 FT MFi Certified Extra Long iPhone Charger Cord. It has a high rating of 4.5 and offers fast charging with a 6-foot length, making it convenient to use while charging. It is Apple MFi Certified, ensuring compatibility and safety for your Apple devices. The cable is durable with reinforced material and a strong connection. It supports fast charging up to 3A/30W and is compatible with a wide range of iPhone and iPad models.\n\nIf you need a multi-device charging solution, the 5 in 1 USB C to Multi Charging Cable 3M/10Ft is also a good option, providing multiple connectors for different devices and the ability to charge three devices simultaneously.\n\nFor a standard iPhone charging cable, the iPhone Charger Cord Lightning Cables 3Pack 3ft Apple MFi Certified is a reliable choice with a 4.5 rating, offering fast charging and durability.\n\nLet me know if y

In [45]:
print(output["answer"])

Based on the available products, I suggest the GREPHONE 2 Pack USB C to Lightning Cable, 6 FT MFi Certified Extra Long iPhone Charger Cord. It has a high rating of 4.5 and offers fast charging with a 6-foot length, making it convenient to use while charging. It is Apple MFi Certified, ensuring compatibility and safety for your Apple devices. The cable is durable with reinforced material and a strong connection. It supports fast charging up to 3A/30W and is compatible with a wide range of iPhone and iPad models.

If you need a multi-device charging solution, the 5 in 1 USB C to Multi Charging Cable 3M/10Ft is also a good option, providing multiple connectors for different devices and the ability to charge three devices simultaneously.

For a standard iPhone charging cable, the iPhone Charger Cord Lightning Cables 3Pack 3ft Apple MFi Certified is a reliable choice with a 4.5 rating, offering fast charging and durability.

Let me know if you want details on any specific cable or have part